# Transverse-Field Ising Model Simulation

This notebook exemplifies how the VQE Lab library can be used for VQE simulations of various models.
It compares two different parametrizations of the same QAOA-like ansatz, where one has directly parametrized entanglers and the other uses fixed entanglers with compensating single-qubit rotations.

In [ ]:
import warnings
from math import pi
from pathlib import Path

import matplotlib.pyplot as plt
from scipy.optimize import OptimizeWarning

import vqe_lab as vqe

ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "workloads").is_dir() and (ROOT / "pyproject.toml").is_file():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError("Could not find the project root.")

warnings.filterwarnings(
    "ignore",
    message="Unknown solver options: iprint",
    category=OptimizeWarning,
)

## Setup
The notebook makes use of the `tfim.py` workload, which defines a parametrized hamiltonian and ansatz.

In [ ]:
model = vqe.load_workload(ROOT / "workloads" / "tfim.py")
N = 4

HAMILTONIAN = {"n": N, "j": 1.0, "h": 1.0}
ANSATZ = {"n": N, "p": 2, "kind": "qaoa"}

vqe.draw_ansatz(model.ansatz(**ANSATZ))

We create an experiment with a combination of Hamiltonian and ansatz and start by evaluating the exact reference ground state energy, vector and half-chain entanglement entropy.

In [ ]:
base = vqe.Experiment(model, hamiltonian=HAMILTONIAN, ansatz=ANSATZ)
exact = base.exact(return_eigenvector=True)
entropy = vqe.entropy(exact.eigenvector)

print(f"Exact ground state energy: {exact.energy:.8f}")
print(f"Half-chain entanglement entropy for this state is: {entropy:.8f} bits.")

## Comparing Parametrized and Fixed Entangling Gates
We run both the parametrized entangler ansatz and the fixed entanler ansatz three times to avoid initialization-related issues. 

In [ ]:
parametrized = vqe.run_grid(
    model,
    hamiltonian=HAMILTONIAN,
    ansatz=ANSATZ,
    ansatz_grid={"parametric": [True]},
    repeats=3,
    seed=123,
    optimizer="lbfgsb",
    maxiter=80,
    optimizer_bounds=(-pi, pi),
)
fixed = vqe.run_grid(
    model,
    hamiltonian=HAMILTONIAN,
    ansatz=ANSATZ,
    ansatz_grid={"parametric": [False]},
    repeats=3,
    seed=123,
    optimizer="lbfgsb",
    maxiter=80,
    optimizer_bounds=(-pi, pi),
)
assert not parametrized.failures and not fixed.failures, (parametrized.failures, fixed.failures)
groups = [("parametrized RZZ", parametrized.runs), ("fixed RZZ", fixed.runs)]

We can now plot the results.
It is possible to define the plots yourself or use one of the VQE Lab's built-in plotting functions.
We start with the overall convergence.

In [ ]:
vqe.plot_convergence(groups, exact=exact)
plt.show()

We can see some initialization-dependent differences in the convergence plots.
However, both versions of the ansatz (fixed and parametric entanglers) should converge similarly.

Next we can inspect the evolution of the entanglement entropy.
This gives us a baseline estimate on what to expect from the parametrized entangler angles.

In [ ]:
vqe.plot_entropy(groups, exact=entropy)
plt.show()

We can then evaluate these values with respect to the gate angles.

In [ ]:
vqe.plot_parameters(groups, absolute=True)
vqe.plot_parameters(groups, heatmap=True, absolute=True)
plt.show()

The first plot shows the evolution of the average of all entangling gate angles over the objective function evaluations.
The heatmap plot compares the per-gate parameters for the best energy.

It is visible that the gate angles converge to a value far below the maximum.

## Comparing different phases
We now want to extend this analysis to different phases of the Ising model.

In [ ]:
HAMILTONIANS = {
    "ferromagnetic": {"n": N, "j": 1.0, "h": 0.2},
    "critical": {"n": N, "j": 1.0, "h": 1.0},
    "paramagnetic": {"n": N, "j": 1.0, "h": 2.0},
}

exact_energies = {}
exact_entropies = {}
results_by_phase = {}
phase_groups = {}
for phase, hamiltonian in HAMILTONIANS.items():
    base = vqe.Experiment(model, hamiltonian=hamiltonian, ansatz=ANSATZ)
    exact_result = base.exact(return_eigenvector=True)
    exact_energies[phase] = exact_result
    exact_entropies[phase] = vqe.entropy(exact_result.eigenvector)

    print(f"Exact ground state energy for the {phase} phase: {exact_result.energy:.8f}")
    print(
        f"Half-chain entanglement entropy for the {phase} phase: "
        f"{exact_entropies[phase]:.8f} bits."
    )
    print()

    results = vqe.run_grid(
        model,
        hamiltonian=hamiltonian,
        ansatz=ANSATZ,
        ansatz_grid={"parametric": [True, False]},
        repeats=3,
        seed=123,
        optimizer="lbfgsb",
        maxiter=80,
    )
    assert not results.failures, results.failures
    results_by_phase[phase] = results

    groups = [
        (
            "parametrized RZZ",
            [run for run in results.runs if run.metadata["ansatz"]["parametric"]],
        ),
        (
            "fixed RZZ",
            [run for run in results.runs if not run.metadata["ansatz"]["parametric"]],
        ),
    ]
    phase_groups[phase] = groups

    vqe.plot_convergence(groups, exact=exact_result)
    plt.title(f"{phase.capitalize()} phase: convergence")
    plt.show()

    vqe.plot_entropy(groups, exact=exact_entropies[phase])
    plt.title(f"{phase.capitalize()} phase: entanglement entropy")
    plt.show()

    vqe.plot_parameters(groups, absolute=True)
    plt.title(f"{phase.capitalize()} phase: average parameter evolution")
    plt.show()

    vqe.plot_parameters(groups, heatmap=True, absolute=True)
    plt.title(f"{phase.capitalize()} phase: converged parameter per gate")
    plt.show()
